# Self-Querying Retriever [Step 2 - Natural Language to Structured Filters]

> **MLCourse - Agentic AI - Agentic RAG**

A self-querying retriever converts a natural language query into structured
metadata filters plus a semantic search. This notebook builds a custom
self-querying system: the LLM extracts filter criteria from the user's
question, then we combine structured filtering with vector similarity search.

In [1]:
import os
import json
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] API key found")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

[GREEN] No API key needed -- using local ChatOllama


### 1. Load the Source Document


In [ ]:
text_path = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"
with open(text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f"Loaded alice.txt: {len(raw_text)} chars")


### 2. Chunk with Metadata


In [ ]:
# We split the text and attach metadata to each chunk: chapter number,
# chunk index, and estimated position in the book (beginning/middle/end).
# This metadata is what the self-querying system will filter on.

from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_text(raw_text)

# Build metadata for each chunk
def build_metadata(chunks):
    """Attach positional and chapter metadata to each chunk."""
    total = len(chunks)
    metadata_list = []
    chapter = 1
    for i, chunk in enumerate(chunks):
        # Heuristic: detect chapter boundaries from text content
        lower = chunk.lower()
        if "chapter" in lower and any(f"chapter {n}" in lower for n in range(1, 15)):
            for n in range(1, 15):
                if f"chapter {n}" in lower:
                    chapter = n
                    break
        # Position in the book
        ratio = i / max(total - 1, 1)
        if ratio < 0.33:
            position = "beginning"
        elif ratio < 0.66:
            position = "middle"
        else:
            position = "end"
        metadata_list.append({
            "chapter": chapter,
            "chunk_index": i,
            "position": position,
        })
    return metadata_list

metadata_list = build_metadata(chunks)
print(f"Built {len(chunks)} chunks with metadata")
print(f"Sample metadata: {metadata_list[0]}")


### 3. Build the Vector Store with Metadata


In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Create Document objects with metadata
docs = [
    Document(page_content=chunks[i], metadata=metadata_list[i])
    for i in range(len(chunks))
]

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="alice_selfquery")
print(f"Vector store built with {vectorstore._collection.count()} documents")


### 4. Initialize the LLM


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("LLM initialized:", llm.model)


### 5. Define the Metadata Schema


In [ ]:
# This tells the LLM what metadata fields are available for filtering.

METADATA_SCHEMA = {
    "chapter": {
        "type": "integer",
        "description": "The chapter number of Alice in Wonderland"
    },
    "chunk_index": {
        "type": "integer",
        "description": "The sequential index of the chunk in the document"
    },
    "position": {
        "type": "string",
        "description": "Position in the book: beginning, middle, or end"
    }
}

schema_description = "\n".join([
    f'  - {k} ({v["type"]}): {v["description"]}'
    for k, v in METADATA_SCHEMA.items()
])
print("Metadata schema:")
print(schema_description)


### 6. Build the Self-Query Extractor


In [ ]:
# The LLM reads the user's natural language query and extracts:
# 1. A structured filter (JSON with metadata conditions)
# 2. A cleaned-up semantic search query

from langchain_core.prompts import ChatPromptTemplate

extractor_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a query parser. Given a user query and available metadata fields, "
     "extract structured filters and a clean search query.\n\n"
     "Available metadata fields:\n{schema}\n\n"
     "OUTPUT FORMAT (JSON only):\n"
     '{{"filters": [{{"field": "...", "operator": "...", "value": ...}}], '
     '"search_query": "cleaned up query for semantic search"}}\n\n'
     "Operators: eq, neq, gt, lt, contains\n"
     "If no metadata filter is needed, use an empty list for filters.\n"
     "If the query already mentions specific metadata (like chapter), extract it.\n"
     "Reply with ONLY the JSON object, no explanation."),
    ("user", "{query}")
])

def extract_filters(query: str) -> dict:
    """Use LLM to extract structured filters from a natural language query."""
    response = (extractor_prompt | llm).invoke({
        "schema": schema_description,
        "query": query
    })
    text = response.content.strip()
    # Strip markdown code fences if present
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
    if text.endswith("```"):
        text = text.rsplit("```", 1)[0]
    text = text.strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        # Fallback: no filters, use original query
        parsed = {"filters": [], "search_query": query}
    return parsed


### 7. Build the Filtering Logic


In [ ]:
# Convert extracted filters into Chroma where clauses.

def build_chroma_filter(filters: list) -> dict:
    """Convert extracted filter list to a Chroma-compatible filter dict."""
    if not filters:
        return None
    conditions = []
    for f in filters:
        field = f.get("field", "")
        op = f.get("operator", "eq")
        val = f.get("value")
        if field not in METADATA_SCHEMA:
            continue
        if op == "eq":
            conditions.append({field: val})
        elif op == "neq":
            conditions.append({field: {"$ne": val}})
        elif op == "gt":
            conditions.append({field: {"$gt": val}})
        elif op == "lt":
            conditions.append({field: {"$lt": val}})
        elif op == "contains":
            conditions.append({field: {"$contains": val}})
    if not conditions:
        return None
    if len(conditions) == 1:
        return conditions[0]
    return {"$and": conditions}


### 8. Build the Self-Query Retriever


In [ ]:
# Combines structured metadata filtering with semantic similarity search.

def self_query_retrieve(query: str, k: int = 4) -> list:
    """Self-query: extract filters, then search with both filters and embeddings."""
    # Step 1: Extract structured filters
    parsed = extract_filters(query)
    search_query = parsed.get("search_query", query)
    filters = parsed.get("filters", [])

    print(f"  [PARSE] Original: {query}")
    print(f"  [PARSE] Search query: {search_query}")
    print(f"  [PARSE] Filters: {json.dumps(filters)}")

    # Step 2: Build Chroma filter
    chroma_filter = build_chroma_filter(filters)

    # Step 3: Search with filters + embeddings
    if chroma_filter:
        print(f"  [SEARCH] Using filter: {chroma_filter}")
        results = vectorstore.similarity_search(search_query, k=k, filter=chroma_filter)
    else:
        print(f"  [SEARCH] No metadata filter, pure semantic search")
        results = vectorstore.similarity_search(search_query, k=k)

    print(f"  [SEARCH] Got {len(results)} results")
    return results


### 9. Test: Pure Semantic Search (No Filters)


In [ ]:
print("=" * 60)
print("TEST 1: Pure semantic search (no metadata filters)")
print("=" * 60)
results = self_query_retrieve("What is the Cheshire Cat known for?")
for i, doc in enumerate(results):
    print(f"\n  Result {i+1} (chapter={doc.metadata.get('chapter')}, pos={doc.metadata.get('position')}):")
    print(f"  {doc.page_content[:150]}...")


### 10. Test: Filtered Search (Chapter-Based)


In [ ]:
print("\n" + "=" * 60)
print("TEST 2: Filtered search (early chapters only)")
print("=" * 60)
results = self_query_retrieve("What does Alice see in the first chapter?", k=3)
for i, doc in enumerate(results):
    print(f"\n  Result {i+1} (chapter={doc.metadata.get('chapter')}, pos={doc.metadata.get('position')}):")
    print(f"  {doc.page_content[:150]}...")


### 11. Test: Position-Based Filter


In [ ]:
print("\n" + "=" * 60)
print("TEST 3: Position-based filter (end of book)")
print("=" * 60)
results = self_query_retrieve("What happens at the end of Alice's adventure?", k=3)
for i, doc in enumerate(results):
    print(f"\n  Result {i+1} (chapter={doc.metadata.get('chapter')}, pos={doc.metadata.get('position')}):")
    print(f"  {doc.page_content[:150]}...")


### 12. Compare: With vs Without Self-Query


In [ ]:
print("\n" + "=" * 60)
print("COMPARISON: Self-query vs pure semantic search")
print("=" * 60)
query = "What happens in the middle of the story with the Queen?"

print("\n--- With self-query (extracts position=middle filter) ---")
sq_results = self_query_retrieve(query, k=2)
for i, doc in enumerate(sq_results):
    print(f"  [{i+1}] chapter={doc.metadata.get('chapter')}, pos={doc.metadata.get('position')}")

print("\n--- Without self-query (pure semantic) ---")
plain_results = vectorstore.similarity_search(query, k=2)
for i, doc in enumerate(plain_results):
    print(f"  [{i+1}] chapter={doc.metadata.get('chapter')}, pos={doc.metadata.get('position')}")


### 13. Inspect a Filter Extraction


In [ ]:
print("\n" + "=" * 60)
print("DETAILED: Filter extraction for a complex query")
print("=" * 60)
complex_query = "Tell me about the tea party scene, but only from chapters after chapter 5"
parsed = extract_filters(complex_query)
print(f"  Query: {complex_query}")
print(f"  Extracted: {json.dumps(parsed, indent=2)}")


### Summary
